In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/PhD/fedlbs_IWCMC/forecast-fedlbs')

In [3]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMNet, GRUNet
from algorithm import fedavg

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [9]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')
else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [4]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 256
n_layers = 2
lr = 0.001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 10 #number of clients to participate in training

# Data Preperation

In [5]:
dataframes = fetch_dataset()

Floor7
Floor3
Floor5
Floor4
Floor6
Floor6
Floor3
Floor5
Floor4
Floor7
Floor21
Floor16
Floor20
Floor21
Floor10
Floor15
Floor20
Floor14
Floor9
Floor17
Floor13
Floor11
Floor18
Floor19
Floor22
Floor8
Floor22
Floor9
Floor8
Floor18
Floor19
Floor14
Floor12
Floor16
Floor17
Floor15
Floor11
Floor13
Floor12
Floor10


In [6]:
n_clients = len(dataframes) #total number of clients to partition data into

In [7]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', '

# Build the training set

In [10]:
train_loader = []
test_loader = []
label_scalers = []
i= 0
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    if i ==10:
      break
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to(device), torch.from_numpy(train_y).to(device))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to(device), torch.from_numpy(test_y).to(device))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y
    i=i+1

input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(68823, 90, 23) (68823, 1)
(40806, 90, 23) (40806, 1)
(72774, 90, 23) (72774, 1)
(40806, 90, 23) (40806, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)


# LSTM model

In [11]:
lstm = LSTMNet(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMNet(
  (lstm): LSTM(23, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (relu): ReLU()
)
814337


## FedAvg

In [12]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = fedavg(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
)


starting avg round 0
clients:  [18  1 19  8 10 17  6 13  4  2]
round 0, starting client 1/10, id: 18
Epoch [1/1], Train Loss: 0.02436254941476661
round 0, starting client 2/10, id: 1
Epoch [1/1], Train Loss: 0.014278632585344774
round 0, starting client 3/10, id: 19
Epoch [1/1], Train Loss: 0.014878077278883186
round 0, starting client 4/10, id: 8
Epoch [1/1], Train Loss: 0.01573214805681405
round 0, starting client 5/10, id: 10
Epoch [1/1], Train Loss: 0.016071597325070852
round 0, starting client 6/10, id: 17
Epoch [1/1], Train Loss: 0.010251868595068735
round 0, starting client 7/10, id: 6
Epoch [1/1], Train Loss: 0.011176020530651195
round 0, starting client 8/10, id: 13
Epoch [1/1], Train Loss: 0.149657403864342
round 0, starting client 9/10, id: 4
Epoch [1/1], Train Loss: 0.012543193384772165
round 0, starting client 10/10, id: 2
Epoch [1/1], Train Loss: 0.014729272607447845
calc smape: 39.512823831971474%
MAE: 12.059530884394144
RMSE: 17.744939768450298
Average Loss:  0.0106055

In [13]:
np.save(f'metrics/fedavg_lstm_outputs_C{num_clients_per_round}.npy', outputs_avg)
np.save(f'metrics/fedavg_lstm_targets_C{num_clients_per_round}.npy', targets_avg)
np.save(f'metrics/fedavg_lstm_loss_C{num_clients_per_round}.npy', loss_avg)
np.save(f'metrics/fedavg_lstm_smape_C{num_clients_per_round}.npy', smape_avg)
np.save(f'metrics/fedavg_lstm_mae_C{num_clients_per_round}.npy', mae_avg)
np.save(f'metrics/fedavg_lstm_rmse_C{num_clients_per_round}.npy', rmse_avg)

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)
